# JiT-S2-VMamba — SSC-bc with adaLN on the FFN ONLY (mixer-only DiM-2)

The `-noadaln` arm removes adaLN from the **whole block**. But SSC only modulates `B`/`C`(/`A`) **inside the SS2D scan**: it can replace the mixer branch's conditioning and nothing else. That run therefore also left the SwiGLU FFN — and the output head — with **no conditioning route at all**, a handicap DiM-2 never asks for and a confound on the measured gap.

This notebook runs the **mixer-only** replacement: `adaln_cond: mlp` keeps adaLN-Zero on the FFN branch and on `FinalLayer`, and turns **only** the mixer branch's `shift/scale/gate` into a condition-independent zero-init bias. `(t, y)` reach the SSM exclusively through `B' = B + b0 + W_B z` and `C' = C + c0 + W_C z`, with DiM-2's dedicated `z = MLP(t, c)`.

Three readings of DiM-2, same `ssc: bc` mixer:

| arm | config | adaLN | params |
|---|---|---|---|
| supplement | `jit-s2-vmamba-ssc-bc.yaml` | both branches | 31.62M |
| **mixer-only (this one)** | `jit-s2-vmamba-ssc-bc-ffnadaln.yaml` | FFN + head | **26.61M** |
| replacement | `jit-s2-vmamba-ssc-bc-noadaln.yaml` | none | 21.01M |

Baseline (adaLN-Zero only) is 31.03M. Params drop because the block's `Linear(384→2304)` becomes `Linear(384→1152)` plus a 1152-wide static bias (−5.31M), partly offset by `ssc_z_mlp` (+0.30M); compute is ~unchanged, since adaLN runs once per **sample**, not per token. Report it as a param-mismatched comparison.

If this arm recovers most of the gap the `-noadaln` run showed, that gap was the FFN losing its conditioning, not SSC failing to condition the SSM — which is the question worth answering.

---

JiT-S2-VMamba on Tiny-ImageNet-200 (64px, patch 8, 200 classes) via the repo's `run_experiment.py` / `evaluate.py`. Comment out the download cell if you attach a dataset instead, and the eval cell if you only want to train.

## 1. Environment  *(Internet ON)*

In [ ]:
import os

# 1) Pin torch to 2.5.1
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu124

# 2) Download wheels with explicit destination
CAUSAL = "causal_conv1d-1.5.0.post8+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
MAMBA  = "mamba_ssm-2.2.4+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"

os.system(f"wget -q https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/{CAUSAL} -O /kaggle/working/{CAUSAL}")
os.system(f"wget -q https://github.com/state-spaces/mamba/releases/download/v2.2.4/{MAMBA} -O /kaggle/working/{MAMBA}")

!pip install -q /kaggle/working/{CAUSAL}
!pip install -q /kaggle/working/{MAMBA}

# 3) Patch mamba-ssm
import glob
for path in glob.glob("/usr/local/lib/python*/dist-packages/mamba_ssm/utils/generation.py"):
    with open(path) as f: src = f.read()
    new = src.replace(
        "from transformers.generation import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput, TextStreamer",
        "from transformers.generation import GenerateDecoderOnlyOutput, TextStreamer",
    ).replace(
        "output_cls = GreedySearchDecoderOnlyOutput if top_k == 1 else SampleDecoderOnlyOutput",
        "output_cls = GenerateDecoderOnlyOutput",
    )
    if new != src:
        with open(path, "w") as f: f.write(new)
        print(f"\u2705 Patched {path}")

print(">>> RESTART RUNTIME NOW <<<")

## 2. Repo  *(clone + cd)*

In [ ]:
# Clone the repo. The SSC vmamba.py (ssc: none|bc|abc), the build_model wiring
# in BOTH run_experiment.py and evaluate.py, the ssc configs, and the tests are
# all on main.
import os
REPO_DIR = "/kaggle/working/thesis_Choustoulakis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Rodamanthosch/thesis_Choustoulakis.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
# sanity: confirm the ssc wiring is present in both entry points
!grep -q ssc scripts/run_experiment.py && grep -q ssc scripts/evaluate.py \
    && echo "wiring OK (train + eval)" || echo "WIRING MISSING -- pull latest main"

## 3. Verify the arm  *(first session only; ~1 min)*

In [ ]:
# === Verify the arm (~1 min; first session only -- comment out on resume).
# 1) equivalence proofs: gate identity exp((s*Delta)A)=exp(Delta*A*s) with B/s,
#    s=1 consistency, bias linearity -- machine epsilon vs selective_scan_ref
# 2) adaln_cond guards: adaln_cond="full" byte-identical to HEAD, identity at
#    init in all 3 modes, ROUTING (the mixer input is invariant to c under
#    "mlp" while the FFN's is not), param accounting, z-MLP routing, guards
# 3) model guards: bit-identity of ssc=none / init semantics / grads / gate /
#    arm exclusivity
# PYTHONPATH=. is REQUIRED: without it "from src.models.vmamba import" fails
# and the tests' try/except misreports it as a missing mamba_ssm (known issue).
# If (1) FAILS after a Kaggle torch/CUDA bump, STOP -- do not train on an
# untrusted kernel build.
!PYTHONPATH=. python tests/test_ssc_equivalence.py
!PYTHONPATH=. python tests/test_adaln_mode_cpu.py
!PYTHONPATH=. python tests/test_ssc_model.py

## 4. Download Tiny-ImageNet  *(to /tmp; comment out if you attach a dataset)*

In [ ]:
# Download Tiny-ImageNet-200 to /tmp (EPHEMERAL: keeps your saved Version small --
# only checkpoints go to /kaggle/working). Re-downloads each session (~2-3 min).
# Faster alternative: attach a Kaggle tiny-imagenet dataset and set DATA_DIR to it,
# then comment this cell out. Requires Internet ON.
import os, zipfile, glob, shutil
DATA_DIR = "/tmp/tiny-imagenet-200"
ZIP      = "/tmp/tiny-imagenet-200.zip"
SRC      = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"

if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    if not os.path.exists(ZIP):
        print("downloading tiny-imagenet-200 (~240MB)...")
        os.system(f"wget -q -O {ZIP} {SRC}")
    print("extracting...")
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/tmp")

# TRAIN: train/<cls>/images/*.JPEG -> train/<cls>/*.JPEG  (ImageFolder layout)
tr = os.path.join(DATA_DIR, "train")
for cls in os.listdir(tr):
    sub = os.path.join(tr, cls, "images")
    if os.path.isdir(sub):
        for f in glob.glob(os.path.join(sub, "*.JPEG")):
            shutil.move(f, os.path.join(tr, cls))
        shutil.rmtree(sub)
    for b in glob.glob(os.path.join(tr, cls, "*_boxes.txt")):
        os.remove(b)

# VAL: flat val/images + val_annotations.txt -> val/<cls>/*.JPEG  (evaluate.py reads val/)
val = os.path.join(DATA_DIR, "val")
ann = os.path.join(val, "val_annotations.txt")
if os.path.exists(ann):
    with open(ann) as f:
        for line in f:
            img, cls = line.split("\t")[:2]
            os.makedirs(os.path.join(val, cls), exist_ok=True)
            s = os.path.join(val, "images", img)
            if os.path.exists(s):
                shutil.move(s, os.path.join(val, cls, img))
    shutil.rmtree(os.path.join(val, "images"), ignore_errors=True)
    os.remove(ann)

print("train classes:", len(glob.glob(tr+"/*")),
      "| val classes:", len(glob.glob(val+"/*")), "| DATA_DIR:", DATA_DIR)

## 5. Settings  *(edits the cloned config so train & eval agree)*

In [ ]:
# ── Settings for THIS notebook ─────────────────────────────────────────
SSC        = "bc"       # this notebook: bc with adaLN on the FFN branch only
EPOCHS     = 100          # full target; resume across sessions until reached
SAVE_FREQ  = 1            # overwrite checkpoint-last.pt EVERY epoch (12h safety)

# Fallback if the download cell was commented out (attached-dataset workflow):
try:
    DATA_DIR
except NameError:
    DATA_DIR = "/kaggle/input/tiny-imagenet/tiny-imagenet-200"  # <-- your attached dataset
    print("DATA_DIR not set by a download cell; using:", DATA_DIR)

CONFIG  = f"configs/tiny_imagenet/jit-s2-vmamba-ssc-{SSC}-ffnadaln.yaml"
OUT_DIR = "/kaggle/working/exp_ssc_bc_ffnadaln"   # persists in the saved Version

# Sync the cloned config so BOTH training and evaluation read identical values
# (evaluate.py has no model CLI overrides -- it builds the model straight from
# config). in_context_len stays 0 and state_init stays none: the arms are
# mutually exclusive (assert in the model). adaln_cond: mlp = adaLN on the FFN
# branch + FinalLayer only; the mixer branch is conditioned ONLY by SSC.
import re
def set_cfg(path, kv):
    s = open(path).read()
    for k, v in kv.items():
        s = re.sub(rf"(?m)^(\s*{k}:).*$", rf"\1 {v}", s, count=1)
    open(path, "w").write(s)
set_cfg(CONFIG, {"data_dir": DATA_DIR, "ssc": SSC, "in_context_len": 0,
                 "state_init": "none", "adaln_cond": "mlp", "ssc_z_mlp": "true"})

# ── Resume across 12h sessions ─────────────────────────────────────
# 1st run: RESUME_INPUT = None. Next session: Save Version, attach THIS notebook's
# previous output as an input, set RESUME_INPUT to ".../exp_ssc_bc_ffnadaln".
RESUME_INPUT = None
import os
RESUME_CKPT = None
for c in ([os.path.join(RESUME_INPUT, "checkpoint-last.pt")] if RESUME_INPUT else []) + \
         [os.path.join(OUT_DIR, "checkpoint-last.pt")]:
    if c and os.path.exists(c):
        RESUME_CKPT = c; break
print("config :", CONFIG, "| ssc:", SSC, "| adaLN: FFN branch + head ONLY (mixer = SSC)")
print("data   :", DATA_DIR, "(exists:", os.path.isdir(DATA_DIR), ")")
print("out    :", OUT_DIR, "| resume:", RESUME_CKPT or "(fresh)")

## 6. Train  *(Save Version before 12h; set RESUME_INPUT next session)*

In [ ]:
# ── TRAIN ──────────────────────────────────────────────────────────────
# data_dir + ssc + adaln_cond now live in the (edited) config; we only override
# run-specific knobs here. Runs in a subprocess (fresh torch, no restart).
CMD = (f"python scripts/run_experiment.py --config {CONFIG} "
       f"checkpoint.output_dir={OUT_DIR} checkpoint.save_last_freq={SAVE_FREQ} "
       f"training.epochs={EPOCHS}")
if RESUME_CKPT:
    CMD += f" checkpoint.resume_from={RESUME_CKPT}"
print(CMD, "\n" + "="*70)
!{CMD}

## 7. Evaluate  *(comment out the whole cell to skip)*

In [ ]:
import glob, os

# Find the checkpoint to evaluate: this session's OUT_DIR first, then any
# attached notebook output (Save Version -> attach it as an input). The
# -noadaln notebooks hardcode a notebook hash here; this searches instead, so
# nothing needs editing after the first Save Version.
RUN = "exp_ssc_bc_ffnadaln"
cands = []
for root in (os.path.dirname(OUT_DIR), "/kaggle/input"):
    for which in ("checkpoint-best.pt", "checkpoint-last.pt"):
        cands += sorted(glob.glob(f"{root}/**/{RUN}/{which}", recursive=True))
assert cands, f"no checkpoint found for {RUN} -- train first, or attach the saved Version"
EVAL_CKPT = cands[0]
EVAL_OUT  = f"/kaggle/working/{RUN}/eval"
print("eval ckpt:", EVAL_CKPT)

EVAL_CMD = (f"python scripts/evaluate.py --config {CONFIG} --checkpoint {EVAL_CKPT} "
            f"--n_samples 10000 --batch_size 200 --ema 1 "
            f"--cfg_scale 2.5 --cfg_interval 0.1 1.0 --out_dir {EVAL_OUT}")
print(EVAL_CMD, "\n" + "="*70)
!PYTHONPATH=. {EVAL_CMD}

## Notes
- Three readings of DiM-2 share the same `ssc: bc` mixer: **supplement** (`-ssc-bc`, adaLN everywhere), **mixer-only** (this notebook, `adaln_cond: mlp`) and **replacement** (`-noadaln`, `adaln_cond: false`). Report all three; the mixer-only arm is the one that isolates *SSC vs adaLN as the route into the SSM*, since SSC cannot condition an FFN.
- adaLN is not deleted anywhere: the removed half becomes a zero-init learned bias, so identity-at-init is preserved exactly (checked in `tests/test_adaln_mode_cpu.py`) and only the conditioning changes, not the optimisation scaffold.
- 26.61M vs the 31.03M baseline: param-mismatched, say so explicitly. Compute is ~unchanged (adaLN is per-sample, not per-token).
- `ssc_z_mlp: true` gives SSC DiM-2's own `z = MLP(t, c)`. It is routed to the **SSC path only** — the surviving FFN adaLN still consumes the raw `c`, so the two conditioning routes stay separable.
- `adaln_cond` / `ssc_z_mlp` now ship on `main`, so the runtime `install_noadaln.sh` patch cell that the `-noadaln` notebooks carry is gone from here (that installer is idempotent and would no-op anyway).
- Keep `EPOCHS`, `cfg_scale`, `DATA_DIR`, seed and sampler identical to every other arm.